# Notebook 09 — Terminal Penalty Sensitivity Analysis (v4 — bug root-cause fixed)

**Paper**: Guéant (2017), §2–3 (terminal condition).

## Root-cause analysis of missing penalty effect

Three potential failure modes were investigated:

1. **`ell_func` ignored** — the terminal condition is correctly wired in `solve_general`:
   `theta_old = -ell_func(|n·Δ|)` at `t=T`. ✓ Not the bug.

2. **Penalty too small vs θ amplitude** — the asymptotic solution θ(t,n) diverges as
   a polynomial in (T−t). For T=1800s, |θ(0,n)| ≫ ℓ(Q) even at α=1.
   The penalty boundary condition is numerically drowned. **This is the main bug.**

3. **Wrong observation variable** — plotting total spread for n≠0 hides the asymmetric
   signal. Bid and ask must be plotted separately. ✓ Fixed in v3, kept here.

## Fix strategy

- **Diagnostic cell**: print θ amplitude vs penalty magnitude at t=T and t=0.
- **Use short horizon T=300s** where θ amplitude is small and penalty is visible.
- **Sweep α up to 50×** half-spread to guarantee the penalty dominates.
- **Plot θ directly** (not only quotes) to confirm the terminal condition propagates.
- **Plot δ_bid and δ_ask separately** for n=+2 and n=−2.

## 0. Imports and calibrated parameters

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append(str(Path().resolve().parents[0]))

from market_making.core.solver_1d import solve_general

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8")
%matplotlib inline

ROOT       = Path.cwd().parents[0]
PARAMS_FILE = ROOT / "data/data/calibrated/calibrated_params.json"

SYMBOLS       = ["AAPL", "SPY", "TSLA"]
DEFAULT_GAMMA = 0.001

assert PARAMS_FILE.exists(), f"Run calibration first — {PARAMS_FILE} not found"

with open(PARAMS_FILE) as f:
    raw_params = json.load(f)

all_params = {}
for symbol in SYMBOLS:
    if symbol not in raw_params:
        print(f"WARNING: {symbol} not in calibrated_params.json")
        continue
    cal = raw_params[symbol]
    all_params[symbol] = dict(
        sigma      = cal["sigma"],
        A          = cal["A"],
        k          = cal["k"],
        Delta      = 1.0,
        Q          = int(cal.get("Q", 4)),
        lot_size   = 1.0,
        _mean_price= cal.get("mean_price", cal["Delta"]),
    )

HALF_SPREAD_REF = {"AAPL": 0.014, "SPY": 0.016, "TSLA": 0.029}

print(f"{'Symbol':<8s} {'sigma':>10s} {'A':>10s} {'k':>8s} {'Q':>4s}")
print("-" * 45)
for sym, p in all_params.items():
    print(f"  {sym:<6s} {p['sigma']:>10.6f} {p['A']:>10.6f} {p['k']:>8.4f} {p['Q']:>4d}")

## 1. Penalty helpers

In [ ]:
def ell_quad(c):
    return lambda q_abs: c * q_abs**2

def ell_linear(c):
    return lambda q_abs: c * q_abs

def make_penalties(params, half_spread_ref, alpha_lin=1.0, alpha_quad=1.0):
    """
    c calibrated so that ell(Q*Delta) = alpha * half_spread_ref.
    """
    q_max  = params["Q"] * params["Delta"]
    c_lin  = alpha_lin  * half_spread_ref / q_max
    c_quad = alpha_quad * half_spread_ref / q_max**2
    return c_lin, c_quad

## 2. DIAGNOSTIC — Why the penalty was invisible

We compare the **amplitude of θ** in the asymptotic regime (t=0)
vs the **penalty magnitude** at max inventory.
If `|θ(0,Q)| ≫ ℓ(Q)`, the penalty is numerically drowned and the curves
will be visually identical despite being mathematically different.

In [ ]:
print(f"{'Symbol':<6s} {'T(s)':>6s} {'|θ(0,Q)|':>12s} {'ℓ(Q) α=1':>12s} "
      f"{'ratio ℓ/θ':>12s} {'α_needed':>12s}")
print("-" * 70)

for symbol in SYMBOLS:
    if symbol not in all_params:
        continue
    params = all_params[symbol]
    Q      = params["Q"]
    hs     = HALF_SPREAD_REF[symbol]

    for T_test in [300, 1800, 7200]:
        sol = solve_general(params, DEFAULT_GAMMA, float(T_test),
                            xi=DEFAULT_GAMMA, N_t=max(300, T_test))
        # θ at t=0 (index 0 after reverse), slot n=+Q (index 2Q)
        theta_0_Q = abs(sol["theta"][0, 2*Q])

        _, c_quad = make_penalties(params, hs, alpha_quad=1.0)
        penalty_Q = c_quad * (Q * params["Delta"])**2  # = hs when alpha=1

        ratio     = penalty_Q / theta_0_Q if theta_0_Q > 0 else float("inf")
        alpha_needed = 1.0 / ratio if ratio > 0 else float("inf")

        print(f"  {symbol:<6s} {T_test:>6d} {theta_0_Q:>12.6f} {penalty_Q:>12.6f} "
              f"{ratio:>12.2e} {alpha_needed:>12.1f}")

print()
print("CONCLUSION: penalty is visible only when alpha_needed ≈ ratio ℓ/θ ≫ 1.")
print("For T=300s, the ratio is much smaller → use T=300s with large alpha.")

## 3. DIAGNOSTIC — Confirm terminal condition propagates in θ

Before plotting quotes, we verify that `ell_func` actually modifies θ at t=T
and that the difference propagates backward.

In [ ]:
symbol = "AAPL"
params = all_params[symbol]
Q      = params["Q"]
hs     = HALF_SPREAD_REF[symbol]
lots   = np.arange(-Q, Q+1, dtype=float)

# Use large alpha to make effect unmistakable
ALPHA_DIAG = 20.0
_, c_quad  = make_penalties(params, hs, alpha_quad=ALPHA_DIAG)

T_diag = 300.0
sol_no = solve_general(params, DEFAULT_GAMMA, T_diag, xi=DEFAULT_GAMMA, N_t=300)
sol_qd = solve_general(params, DEFAULT_GAMMA, T_diag, xi=DEFAULT_GAMMA, N_t=300,
                       ell_func=ell_quad(c_quad))

expected_terminal = np.array([-c_quad * (n * params["Delta"])**2 for n in lots])

print("=== theta at t=T (index -1 after reverse) ===")
print(f"  {'n':>4s}  {'no penalty':>14s}  {'quadratic':>14s}  {'expected':>14s}  {'diff':>14s}")
for i, n in enumerate(lots):
    th_no = sol_no["theta"][-1, i]
    th_qd = sol_qd["theta"][-1, i]
    exp   = expected_terminal[i]
    print(f"  {int(n):>4d}  {th_no:>14.8f}  {th_qd:>14.8f}  {exp:>14.8f}  {th_qd-th_no:>14.8f}")

print()
print("=== theta at t=0 (index 0) ===")
print(f"  {'n':>4s}  {'no penalty':>14s}  {'quadratic':>14s}  {'diff':>14s}")
for i, n in enumerate(lots):
    th_no = sol_no["theta"][0, i]
    th_qd = sol_qd["theta"][0, i]
    print(f"  {int(n):>4d}  {th_no:>14.8f}  {th_qd:>14.8f}  {th_qd-th_no:>14.8f}")

print()
print("If diff at t=T is non-zero → ell_func correctly applied.")
print("If diff at t=0 is non-zero → boundary condition propagates backward. ✓")

## 4. Penalty calibration — effective alpha

Based on the diagnostic above, we pick an `alpha` large enough that the penalty
is comparable to `|θ(0, Q)|`.  
We test three regimes: **weak** (α=1), **moderate** (α=10), **strong** (α=50).

In [ ]:
# Short horizon where penalty is most visible
T_SHORT  = 300.0
N_T_SHORT = 300

# Long horizon to show asymptotic negligibility
T_LONG  = 7200.0
N_T_LONG = 3600

ALPHAS_SWEEP = [1.0, 10.0, 50.0]

## A1. θ near T — confirm penalty propagation visually

Plot θ(t, n=+Q) directly over time for T=300s.  
The terminal value at t=T must diverge between ℓ≡0 and ℓ=cq²,  
and the difference must decay as t moves away from T.

In [ ]:
fig, axes = plt.subplots(1, len(all_params), figsize=(8*len(all_params), 5))
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

for ax, symbol in zip(axes, all_params.keys()):
    params = all_params[symbol]
    Q      = params["Q"]
    hs     = HALF_SPREAD_REF[symbol]
    i_Q    = 2 * Q   # index of n=+Q

    sol_no = solve_general(params, DEFAULT_GAMMA, T_SHORT,
                           xi=DEFAULT_GAMMA, N_t=N_T_SHORT)
    times  = sol_no["times"]

    ax.plot(times, sol_no["theta"][:, i_Q], lw=2.5, color="black", label="ℓ ≡ 0")

    colors = ["#2563EB", "#16A34A", "#DC2626"]
    for alpha, clr in zip(ALPHAS_SWEEP, colors):
        _, cq = make_penalties(params, hs, alpha_quad=alpha)
        sol   = solve_general(params, DEFAULT_GAMMA, T_SHORT,
                              xi=DEFAULT_GAMMA, N_t=N_T_SHORT,
                              ell_func=ell_quad(cq))
        ax.plot(times, sol["theta"][:, i_Q], lw=1.8, ls="--", color=clr,
                label=f"α={alpha}")

    ax.set_xlabel("t (s)")
    ax.set_ylabel(f"θ(t, n=+{Q})")
    ax.set_title(f"{symbol} — θ at max inventory (T={T_SHORT:.0f}s)",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "A1: θ(t, n=+Q) for T=300s — penalty modifies terminal value,\n"
    "difference decays backward (larger α → more visible)",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

## A2. δ_bid at n=0 — penalty effect over time (T=300s)

For n=0, the penalty raises δ_bid near T:  
the market maker avoids **opening** a long position just before maturity.  
This is inventory-aversion, not spread tightening.

In [ ]:
fig, axes = plt.subplots(1, len(all_params), figsize=(8*len(all_params), 5))
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

for ax, symbol in zip(axes, all_params.keys()):
    params = all_params[symbol]
    Q      = params["Q"]
    hs     = HALF_SPREAD_REF[symbol]
    i_zero = Q   # n=0

    sol_no = solve_general(params, DEFAULT_GAMMA, T_SHORT,
                           xi=DEFAULT_GAMMA, N_t=N_T_SHORT)
    times  = sol_no["times"]

    ax.plot(times, sol_no["delta_bid"][:, i_zero], lw=2.5, color="black",
            label="ℓ ≡ 0")

    colors = ["#2563EB", "#16A34A", "#DC2626"]
    for alpha, clr in zip(ALPHAS_SWEEP, colors):
        _, cq = make_penalties(params, hs, alpha_quad=alpha)
        sol   = solve_general(params, DEFAULT_GAMMA, T_SHORT,
                              xi=DEFAULT_GAMMA, N_t=N_T_SHORT,
                              ell_func=ell_quad(cq))
        ax.plot(times, sol["delta_bid"][:, i_zero], lw=1.8, ls="--", color=clr,
                label=f"α={alpha}")

    ax.set_xlabel("t (s)")
    ax.set_ylabel("δ_bid(t, n=0)")
    ax.set_title(f"{symbol} — bid quote at n=0 (T={T_SHORT:.0f}s)",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "A2: Near T, penalty raises δ_bid at n=0 → market maker avoids opening new positions",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

## A3. Spread and skew at t=0 — long horizon T=2h

At t=0 with T=7200s, the asymptotic regime dominates.  
All penalty curves coincide with ℓ≡0, validating the paper's simplification.

In [ ]:
n_syms = len(all_params)
fig, axes = plt.subplots(n_syms, 2, figsize=(14, 5*n_syms))
if n_syms == 1:
    axes = axes.reshape(1, 2)

for row, symbol in enumerate(all_params.keys()):
    params = all_params[symbol]
    Q      = params["Q"]
    hs     = HALF_SPREAD_REF[symbol]

    sol_no = solve_general(params, DEFAULT_GAMMA, T_LONG,
                           xi=DEFAULT_GAMMA, N_t=N_T_LONG)
    lots   = sol_no["lots"]
    colors = ["#2563EB", "#16A34A", "#DC2626"]

    for col, (ylabel, get_y) in enumerate([
        ("Spread(0,n)", lambda s: s["delta_bid"][0,:] + s["delta_ask"][0,:]),
        ("Skew(0,n)",   lambda s: s["delta_bid"][0,:] - s["delta_ask"][0,:]),
    ]):
        ax = axes[row, col]
        y  = get_y(sol_no)
        m  = np.isfinite(y)
        ax.plot(lots[m], y[m], lw=2.5, color="black", label="ℓ ≡ 0")

        for alpha, clr in zip(ALPHAS_SWEEP, colors):
            _, cq = make_penalties(params, hs, alpha_quad=alpha)
            sol   = solve_general(params, DEFAULT_GAMMA, T_LONG,
                                  xi=DEFAULT_GAMMA, N_t=N_T_LONG,
                                  ell_func=ell_quad(cq))
            y2 = get_y(sol)
            ax.plot(lots[m], y2[m], lw=1.8, ls="--", color=clr, label=f"α={alpha}")

        ax.set_xlabel("n (lots)")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{symbol} — {ylabel} at t=0 (T=7200s)",
                     fontsize=13, fontweight="bold")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

plt.suptitle(
    "A3: At t=0 with T=2h, all curves collapse → asymptotic regime dominates, ℓ≡0 justified",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

## A4. Penalty decay with horizon

Max |δ(ℓ≠0) − δ(ℓ=0)| at t=0 as a function of T.  
Uses a fixed large alpha (α=20) to keep the effect visible across all horizons.

In [ ]:
T_values   = [60, 120, 300, 600, 1200, 1800, 3600, 7200]
ALPHA_DECAY = 20.0

fig, axes = plt.subplots(1, len(all_params), figsize=(8*len(all_params), 5))
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

for ax, symbol in zip(axes, all_params.keys()):
    params = all_params[symbol]
    hs     = HALF_SPREAD_REF[symbol]
    c_lin, c_quad = make_penalties(params, hs,
                                   alpha_lin=ALPHA_DECAY, alpha_quad=ALPHA_DECAY)
    errors_lin, errors_quad = [], []

    for T_test in T_values:
        nt = max(60, T_test)
        s_no  = solve_general(params, DEFAULT_GAMMA, float(T_test),
                              xi=DEFAULT_GAMMA, N_t=nt)
        s_lin = solve_general(params, DEFAULT_GAMMA, float(T_test),
                              xi=DEFAULT_GAMMA, N_t=nt,
                              ell_func=ell_linear(c_lin))
        s_qd  = solve_general(params, DEFAULT_GAMMA, float(T_test),
                              xi=DEFAULT_GAMMA, N_t=nt,
                              ell_func=ell_quad(c_quad))

        db_no  = s_no["delta_bid"][0,:]
        db_lin = s_lin["delta_bid"][0,:]
        db_qd  = s_qd["delta_bid"][0,:]

        m_lin = np.isfinite(db_no) & np.isfinite(db_lin)
        m_qd  = np.isfinite(db_no) & np.isfinite(db_qd)

        errors_lin.append(np.max(np.abs(db_no[m_lin] - db_lin[m_lin])) if m_lin.any() else 0)
        errors_quad.append(np.max(np.abs(db_no[m_qd]  - db_qd[m_qd]))  if m_qd.any()  else 0)

    ax.plot(T_values, errors_lin,  "o-",  lw=2, ms=7, label=f"Linear (α={ALPHA_DECAY})")
    ax.plot(T_values, errors_quad, "s--", lw=2, ms=7, label=f"Quadratic (α={ALPHA_DECAY})")
    ax.set_xlabel("T (s)")
    ax.set_ylabel("max |δ(ℓ≠0) − δ(ℓ=0)| at t=0")
    ax.set_title(f"{symbol} — penalty decay with horizon",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_yscale("log")

plt.suptitle(
    f"A4: Penalty effect at t=0 decays with T (α={ALPHA_DECAY}) — negligible for T > 1h",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

## A5. Asymmetric bid/ask dynamics near T — n = ±2  

**The key result.** For n≠0, the penalty acts asymmetrically:

| Inventory | δ_ask | δ_bid | Economic meaning |
|-----------|-------|-------|------------------|
| n = +2 (long) | ↓ falls | ↑ rises | sell faster, stop buying |
| n = −2 (short) | ↑ rises | ↓ falls | stop selling, buy back faster |

Total spread = δ_bid + δ_ask is the **wrong metric** — it can go either way.
We plot δ_bid and δ_ask **separately**.

In [ ]:
ALPHA_ASYM = 20.0   # large enough to be visible

fig, axes = plt.subplots(len(all_params), 4,
                         figsize=(6*4, 5*len(all_params)))
if len(all_params) == 1:
    axes = axes.reshape(1, 4)

for row, symbol in enumerate(all_params.keys()):
    params = all_params[symbol]
    Q      = params["Q"]
    hs     = HALF_SPREAD_REF[symbol]
    _, cq  = make_penalties(params, hs, alpha_quad=ALPHA_ASYM)

    sol_no = solve_general(params, DEFAULT_GAMMA, T_SHORT,
                           xi=DEFAULT_GAMMA, N_t=N_T_SHORT)
    sol_qd = solve_general(params, DEFAULT_GAMMA, T_SHORT,
                           xi=DEFAULT_GAMMA, N_t=N_T_SHORT,
                           ell_func=ell_quad(cq))
    times  = sol_no["times"]

    n_pos = min(2, Q);  i_pos = n_pos + Q
    n_neg = -min(2, Q); i_neg = n_neg + Q

    configs = [
        (i_pos, "delta_bid", f"δ_bid(t, n=+{n_pos})",
         "Expected: ↑ rises near T\n(avoid buying more)"),
        (i_pos, "delta_ask", f"δ_ask(t, n=+{n_pos})",
         "Expected: ↓ falls near T\n(sell more aggressively)"),
        (i_neg, "delta_bid", f"δ_bid(t, n={n_neg})",
         "Expected: ↓ falls near T\n(buy back more aggressively)"),
        (i_neg, "delta_ask", f"δ_ask(t, n={n_neg})",
         "Expected: ↑ rises near T\n(avoid selling more)"),
    ]

    for col, (i_lot, key, ylabel, note) in enumerate(configs):
        ax    = axes[row, col]
        y_no  = sol_no[key][:, i_lot]
        y_qd  = sol_qd[key][:, i_lot]

        ax.plot(times, y_no, lw=2.5, color="black",  label="ℓ ≡ 0")
        ax.plot(times, y_qd, lw=2,   color="#DC2626", ls="--",
                label=f"ℓ=cq² (α={ALPHA_ASYM})")

        ax.axvline(0.85 * T_SHORT, ls=":", color="gray", alpha=0.5)
        ax.text(0.02, 0.97, note, transform=ax.transAxes,
                fontsize=8, va="top",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8))

        ax.set_xlabel("t (s)")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{symbol} — {ylabel}", fontsize=11, fontweight="bold")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle(
    f"A5: Asymmetric inventory-flattening near T (α={ALPHA_ASYM}, T={T_SHORT:.0f}s)\n"
    "Long (n=+2): aggressive ask ↓, passive bid ↑  |  Short (n=−2): aggressive bid ↓, passive ask ↑",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

print("Penalty acts ASYMMETRICALLY: it does not uniformly tighten the spread.")
print("It distorts bid and ask in opposite directions depending on inventory sign.")

## A6. Penalty strength sweep — δ_ask for n=+2

Sweep α ∈ {1, 10, 20, 50} to show monotone liquidation pressure.  
Stronger penalty → earlier and deeper fall of δ_ask.

In [ ]:
ALPHAS_FULL = [1.0, 10.0, 20.0, 50.0]

fig, axes = plt.subplots(1, len(all_params), figsize=(8*len(all_params), 5))
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

for ax, symbol in zip(axes, all_params.keys()):
    params = all_params[symbol]
    Q      = params["Q"]
    hs     = HALF_SPREAD_REF[symbol]
    n_pos  = min(2, Q)
    i_lot  = n_pos + Q

    sol_no = solve_general(params, DEFAULT_GAMMA, T_SHORT,
                           xi=DEFAULT_GAMMA, N_t=N_T_SHORT)
    times  = sol_no["times"]
    ax.plot(times, sol_no["delta_ask"][:, i_lot],
            lw=2.5, color="black", label="ℓ ≡ 0")

    colors = ["#BFDBFE", "#2563EB", "#16A34A", "#DC2626"]
    for alpha, clr in zip(ALPHAS_FULL, colors):
        _, cq = make_penalties(params, hs, alpha_quad=alpha)
        sol   = solve_general(params, DEFAULT_GAMMA, T_SHORT,
                              xi=DEFAULT_GAMMA, N_t=N_T_SHORT,
                              ell_func=ell_quad(cq))
        ax.plot(times, sol["delta_ask"][:, i_lot],
                lw=1.8, ls="--", color=clr,
                label=f"α={alpha} (ℓ(Q)={alpha*hs:.3f})")

    ax.set_xlabel("t (s)")
    ax.set_ylabel(f"δ_ask(t, n=+{n_pos})")
    ax.set_title(f"{symbol} — ask quote sweep (T={T_SHORT:.0f}s, n=+{n_pos})",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f"A6: Stronger α → earlier/steeper fall of δ_ask (T={T_SHORT:.0f}s, n=+{n_pos})\n"
    "Monotone liquidation pressure — confirms inventory-flattening mechanism",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

## Key Takeaways

### Root cause of missing effect (fixed in v4)

> For T ≥ 1800s, the amplitude of θ(0, n) in the asymptotic regime is several
> orders of magnitude larger than ℓ(Q·Δ) at α=1. The penalty is applied
> correctly in the solver but is numerically invisible at t=0 for long horizons.
> Fix: use T=300s and α ≥ 10 to bring the penalty into the same scale as θ.

### Economic interpretation (corrected from v1–v2)

1. **Near T, n=0**: penalty raises δ_bid and δ_ask symmetrically → market maker
   avoids *opening* any position close to maturity ("stay flat").

2. **Near T, n=+2**: penalty lowers δ_ask (sell aggressively) and raises δ_bid
   (stop buying) → **asymmetric distortion** toward inventory reduction.

3. **Near T, n=−2**: mirror image — δ_bid falls, δ_ask rises.

4. **Total spread** is the wrong metric for n≠0: it can increase, decrease or
   stay flat depending on which asymmetric effect dominates.

5. **For T > 1h**: penalty effect at t=0 decays to negligible levels,
   validating the paper's choice ℓ ≡ 0 for intraday horizons.

### Report paragraph

> The terminal penalty does not uniformly tighten the spread. Instead, it
> distorts bid and ask quotes **asymmetrically** depending on the sign of the
> current inventory. Close to maturity, a market maker with positive inventory
> posts a more aggressive ask (lower δ_ask) to accelerate sales, and a less
> aggressive bid (higher δ_bid) to avoid increasing the position further.
> The economically relevant effect is an **inventory-dependent distortion of
> the quoting policy** toward terminal inventory reduction — not a uniform
> reduction of the spread. For horizons exceeding one hour, the boundary
> condition's influence decays to negligible levels, validating the paper's
> simplification ℓ ≡ 0.